# AI-Generated Hotel Review Detector — Final Demo

End-to-end demo on seven curated reviews using `FinalReviewDetector`.

In [3]:
import json
import sys
from pathlib import Path

notebook_dir = Path.cwd()
repo_root = None
for candidate in [notebook_dir, *notebook_dir.parents]:
    if (candidate / "src" / "final_detector.py").exists():
        repo_root = candidate
        break
if repo_root is None:
    raise RuntimeError("Could not locate src/final_detector.py from the notebook's working directory.")

sys.path.insert(0, str(repo_root))

from src.final_detector import FinalReviewDetector
from src.stylometry_features import extract_stylometry_features

detector = FinalReviewDetector()
print(f"Detector loaded.")
print(f"Selected model:  {detector.selected_model_name}")

Detector loaded.
Selected model:  random_forest


Helper that runs the detector and prints structured output.

In [4]:
def show(label, text):
    print("=" * 80)
    print(label)
    print("-" * 80)
    print(f"Review: {text}")
    print("-" * 80)

    features = extract_stylometry_features(text)
    result = detector.detect_review(
        review_text=text,
        extracted_features=features,
        use_calibration=True,
    )

    print(f"  model_used:        {result.model_used}")
    print(f"  calibrated:        {result.calibrated}")
    print(f"  ai_probability:    {result.ai_probability}")
    print(f"  ai_likeness_score: {result.ai_likeness_score}")
    print(f"  uncertainty_band:  {result.uncertainty_band}")
    print(f"  predicted_label:   {result.predicted_label}")
    print(f"  top_features:      {result.top_features}")
    print(f"  explanation:")
    print(f"    {result.explanation}")
    print()
    return result

**Example 1** — polished AI-style review (expect score ≈ 100).

In [5]:
r1 = show(
    "Example 1: AI-style polished review",
    "This hotel exceeded all expectations. The staff went above and beyond to ensure our stay was memorable, and every amenity was thoughtfully arranged for comfort and convenience."
)

Example 1: AI-style polished review
--------------------------------------------------------------------------------
Review: This hotel exceeded all expectations. The staff went above and beyond to ensure our stay was memorable, and every amenity was thoughtfully arranged for comfort and convenience.
--------------------------------------------------------------------------------
  model_used:        random_forest
  calibrated:        True
  ai_probability:    0.9999
  ai_likeness_score: 100
  uncertainty_band:  likely AI-generated
  predicted_label:   AI
  top_features:      {'capital_letter_ratio': 0.0114, 'stopword_ratio': 0.333, 'avg_word_length': 5.444}
  explanation:
    The detector assigns an AI probability of 0.9999 with an AI-likeness score of 100, which falls in the 'likely AI-generated' band. The prediction is driven by the top stylometric features capital_letter_ratio=0.0114, stopword_ratio=0.333, avg_word_length=5.444.



**Example 2** — AI review with generator prompt-echo prefix (stripped before inference).

In [9]:
r2 = show(
    "Example 2: AI review with prompt-echo prefix",
    "Okay, here's a TripAdvisor-style hotel review following your guidelines: We absolutely loved our stay at The Willow Creek Inn! The rooms were comfortable and clean, and the staff were incredibly friendly. "
    "The breakfast spread was exceptional with fresh fruit and a proper full English option. Definitely recommend it for a budget-friendly stay."
)

Example 2: AI review with prompt-echo prefix
--------------------------------------------------------------------------------
Review: Okay, here's a TripAdvisor-style hotel review following your guidelines: We absolutely loved our stay at The Willow Creek Inn! The rooms were comfortable and clean, and the staff were incredibly friendly. The breakfast spread was exceptional with fresh fruit and a proper full English option. Definitely recommend it for a budget-friendly stay.
--------------------------------------------------------------------------------
  model_used:        random_forest
  calibrated:        True
  ai_probability:    1.0
  ai_likeness_score: 100
  uncertainty_band:  likely AI-generated
  predicted_label:   AI
  top_features:      {'capital_letter_ratio': 0.0349, 'stopword_ratio': 0.327, 'avg_word_length': 5.145}
  explanation:
    The detector assigns an AI probability of 1.0000 with an AI-likeness score of 100, which falls in the 'likely AI-generated' band. The predic

**Example 3** — short AI-style review.

In [10]:
r3 = show(
    "Example 3: Short, terse AI-style review",
    "An exceptional stay. The service was impeccable, the room beautifully appointed, and the location truly unbeatable. Highly recommended."
)

Example 3: Short, terse AI-style review
--------------------------------------------------------------------------------
Review: An exceptional stay. The service was impeccable, the room beautifully appointed, and the location truly unbeatable. Highly recommended.
--------------------------------------------------------------------------------
  model_used:        random_forest
  calibrated:        True
  ai_probability:    0.9996
  ai_likeness_score: 100
  uncertainty_band:  likely AI-generated
  predicted_label:   AI
  top_features:      {'capital_letter_ratio': 0.0222, 'stopword_ratio': 0.333, 'avg_word_length': 6.278}
  explanation:
    The detector assigns an AI probability of 0.9996 with an AI-likeness score of 100, which falls in the 'likely AI-generated' band. The prediction is driven by the top stylometric features capital_letter_ratio=0.0222, stopword_ratio=0.333, avg_word_length=6.278.



**Example 4** — casual lowercased human review (in-distribution; expect Human).

In [11]:
r4 = show(
    "Example 4: Casual lower-cased human review",
    "stayed here last week. rooms ok but wifi terrible. breakfast was meh. location good though. would return if price drops"
)

Example 4: Casual lower-cased human review
--------------------------------------------------------------------------------
Review: stayed here last week. rooms ok but wifi terrible. breakfast was meh. location good though. would return if price drops
--------------------------------------------------------------------------------
  model_used:        random_forest
  calibrated:        True
  ai_probability:    0.0
  ai_likeness_score: 0
  uncertainty_band:  likely human-written
  predicted_label:   Human
  top_features:      {'capital_letter_ratio': 0.0, 'stopword_ratio': 0.15, 'avg_word_length': 4.8}
  explanation:
    The detector assigns an AI probability of 0.0000 with an AI-likeness score of 0, which falls in the 'likely human-written' band. The prediction is supported by the top stylometric features capital_letter_ratio=0.0, stopword_ratio=0.15, avg_word_length=4.8.



**Example 5** — long sentence-cased human review (OOD; expect misclassification).

In [12]:
r5 = show(
    "Example 5: Long, descriptive human-style review",
    "We stayed here for four nights with two kids and a grandparent. The double-bed family room was tighter than the photos suggested but it worked. "
    "Walls are thin enough that we heard the hallway clearly during checkouts at 6am. Breakfast was hit and miss - the coffee machine kept breaking on day three and the pastries on day four were clearly from the previous morning. "
    "That said, the location is unbeatable - five minutes to the metro and ten to the old town. Staff were patient with our broken Italian. Would go back if the price was right but I would not pay rack rate."
)

Example 5: Long, descriptive human-style review
--------------------------------------------------------------------------------
Review: We stayed here for four nights with two kids and a grandparent. The double-bed family room was tighter than the photos suggested but it worked. Walls are thin enough that we heard the hallway clearly during checkouts at 6am. Breakfast was hit and miss - the coffee machine kept breaking on day three and the pastries on day four were clearly from the previous morning. That said, the location is unbeatable - five minutes to the metro and ten to the old town. Staff were patient with our broken Italian. Would go back if the price was right but I would not pay rack rate.
--------------------------------------------------------------------------------


  model_used:        random_forest
  calibrated:        True
  ai_probability:    1.0
  ai_likeness_score: 100
  uncertainty_band:  likely AI-generated
  predicted_label:   AI
  top_features:      {'capital_letter_ratio': 0.0158, 'stopword_ratio': 0.394, 'avg_word_length': 4.385}
  explanation:
    The detector assigns an AI probability of 1.0000 with an AI-likeness score of 100, which falls in the 'likely AI-generated' band. The prediction is driven by the top stylometric features capital_letter_ratio=0.0158, stopword_ratio=0.394, avg_word_length=4.385.



**Example 6** — ambiguous mixed review (uncertain band should activate).

In [13]:
r6 = show(
    "Example 6: Ambiguous mixed review",
    "The hotel was fine. Clean room, decent service, good location. Nothing special but nothing bad either."
)

Example 6: Ambiguous mixed review
--------------------------------------------------------------------------------
Review: The hotel was fine. Clean room, decent service, good location. Nothing special but nothing bad either.
--------------------------------------------------------------------------------


  model_used:        random_forest
  calibrated:        True
  ai_probability:    0.4237
  ai_likeness_score: 42
  uncertainty_band:  uncertain
  predicted_label:   Human
  top_features:      {'capital_letter_ratio': 0.0294, 'stopword_ratio': 0.188, 'avg_word_length': 5.125}
  explanation:
    The detector assigns an AI probability of 0.4237 with an AI-likeness score of 42, which falls in the 'uncertain' band. The prediction is supported by the top stylometric features capital_letter_ratio=0.0294, stopword_ratio=0.188, avg_word_length=5.125.



**Example 7** — sanity-check failure: clean human-style review classified as AI.

In [14]:
r7 = show(
    "Example 7: Sanity-check failure (clean human-style review classified as AI)",
    "The room was clean and the location was convenient, but the breakfast was disappointing and the walls were thin."
)

Example 7: Sanity-check failure (clean human-style review classified as AI)
--------------------------------------------------------------------------------
Review: The room was clean and the location was convenient, but the breakfast was disappointing and the walls were thin.
--------------------------------------------------------------------------------


  model_used:        random_forest
  calibrated:        True
  ai_probability:    0.9999
  ai_likeness_score: 100
  uncertainty_band:  likely AI-generated
  predicted_label:   AI
  top_features:      {'capital_letter_ratio': 0.0089, 'stopword_ratio': 0.579, 'avg_word_length': 4.842}
  explanation:
    The detector assigns an AI probability of 0.9999 with an AI-likeness score of 100, which falls in the 'likely AI-generated' band. The prediction is driven by the top stylometric features capital_letter_ratio=0.0089, stopword_ratio=0.579, avg_word_length=4.842.



Summary table.

In [15]:
import pandas as pd

def row(label, r):
    return {
        "example": label,
        "ai_likeness_score": r.ai_likeness_score,
        "uncertainty_band": r.uncertainty_band,
        "predicted_label": r.predicted_label,
    }

summary = pd.DataFrame([
    row("1. AI-style polished review",          r1),
    row("2. AI review with prompt-echo prefix", r2),
    row("3. short terse AI-style review",       r3),
    row("4. casual lower-cased human review",   r4),
    row("5. long descriptive human review",     r5),
    row("6. ambiguous mixed review",            r6),
    row("7. sanity-check failure case",         r7),
])
summary

,example,ai_likeness_score,uncertainty_band,predicted_label
0,1. AI-style polished review,100,likely AI-generated,AI
1,2. AI review with prompt-echo prefix,100,likely AI-generated,AI
2,3. short terse AI-style review,100,likely AI-generated,AI
3,4. casual lower-cased human review,0,likely human-written,Human
4,5. long descriptive human review,100,likely AI-generated,AI
5,6. ambiguous mixed review,42,uncertain,Human
6,7. sanity-check failure case,100,likely AI-generated,AI


Interactive use:

In [17]:
my_review = (
    "We had a comfortable two-night stay. Beds were soft and the lobby was nice. Breakfast was just okay."
)

features = extract_stylometry_features(my_review)
result = detector.detect_review_dict(
    review_text=my_review,
    extracted_features=features,
)
print(json.dumps(result, indent=2))

{
  "review_text": "We had a comfortable two-night stay. Beds were soft and the lobby was nice. Breakfast was just okay.",
  "model_used": "random_forest",
  "calibrated": true,
  "ai_probability": 1.0,
  "ai_likeness_score": 100,
  "uncertainty_band": "likely AI-generated",
  "predicted_label": "AI",
  "top_features": {
    "capital_letter_ratio": 0.03,
    "stopword_ratio": 0.421,
    "avg_word_length": 4.158
  },
  "explanation": "The detector assigns an AI probability of 1.0000 with an AI-likeness score of 100, which falls in the 'likely AI-generated' band. The prediction is driven by the top stylometric features capital_letter_ratio=0.03, stopword_ratio=0.421, avg_word_length=4.158."
}
